In [8]:
import os
import torch
import json
import shutil
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from PIL import Image

# AI Libraries
from ultralytics import YOLO
from anomalib.data import Folder
from anomalib.models import Patchcore
from anomalib.engine import Engine
from anomalib.deploy import TorchInferencer

# --- 1. PATH CONFIGURATION FOR *COMBINED* DATA ---
PROJECT_ROOT = Path("E:/Project-Work")

# Inputs for TOP 1
RAW_DATA_DIR_1 = PROJECT_ROOT / "Data/TOP 1"
LABELS_CSV_1 = PROJECT_ROOT / "Preprocessing/features_labeled.csv"

# Inputs for TOP 2
RAW_DATA_DIR_2 = PROJECT_ROOT / "Data/TOP 2"
LABELS_CSV_2 = PROJECT_ROOT / "Preprocessing/features_labeled_2.csv"

# Common Inputs
# --- FIX: Define BOTH config paths ---
ROI_CONFIG_PATH_1 = PROJECT_ROOT / "Preprocessing/roi_config.json" # The OLD one for TOP 1
ROI_CONFIG_PATH_2 = PROJECT_ROOT / "Preprocessing/connector_crops.json" # The NEW one for TOP 2
# --- END FIX ---
REFERENCE_IMG_NAME = "20251106131917_TOP.png" 

# Outputs for COMBINED Data (using original paths)
ALIGNED_DIR = PROJECT_ROOT / "Data/aligned_top"
CROPS_DIR = PROJECT_ROOT / "Data/connectors" 
PREPROCESSED_CROPS_DIR = PROJECT_ROOT / "Data/connectors_preprocessed"
FINAL_DATASET_DIR = PROJECT_ROOT / "Final_Dataset" 

# Make sure outputs exist
for d in [ALIGNED_DIR, CROPS_DIR, PREPROCESSED_CROPS_DIR, FINAL_DATASET_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f" Project Root set to: {PROJECT_ROOT}")
print(f"--- PROCESSING COMBINED 'TOP 1' + 'TOP 2' DATASET --- ")
print(f" Reading images from: {RAW_DATA_DIR_1} and {RAW_DATA_DIR_2}")
print(f" Reading labels from: {LABELS_CSV_1} and {LABELS_CSV_2}")
print(f" Reading TOP 1 ROI config from: {ROI_CONFIG_PATH_1}")
print(f" Reading TOP 2 ROI config from: {ROI_CONFIG_PATH_2}")

 Project Root set to: E:\Project-Work
--- PROCESSING COMBINED 'TOP 1' + 'TOP 2' DATASET --- 
 Reading images from: E:\Project-Work\Data\TOP 1 and E:\Project-Work\Data\TOP 2
 Reading labels from: E:\Project-Work\Preprocessing\features_labeled.csv and E:\Project-Work\Preprocessing\features_labeled_2.csv
 Reading TOP 1 ROI config from: E:\Project-Work\Preprocessing\roi_config.json
 Reading TOP 2 ROI config from: E:\Project-Work\Preprocessing\connector_crops.json


In [9]:
def load_image(path):
    path = str(path)
    img = cv2.imread(path)
    if img is None: raise FileNotFoundError(f"Cannot read: {path}")
    return img

def align_image(img, ref_img, ref_gray, crop_coords):
    """Aligns board to reference and crops the background"""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # ORB Features
    orb = cv2.ORB_create(nfeatures=5000)
    kp1, des1 = orb.detectAndCompute(ref_gray, None)
    kp2, des2 = orb.detectAndCompute(gray, None)

    # Match Features
    matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = matcher.match(des1, des2)
    matches = sorted(matches, key=lambda x: x.distance)

    # Keep top 15% matches
    good_matches = matches[:int(len(matches) * 0.15)]
    if len(good_matches) < 10: return None

    # Extract points
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)

    # Find Homography & Warp
    H, _ = cv2.findHomography(dst_pts, src_pts, cv2.RANSAC, 5.0)
    h, w = ref_img.shape[:2]
    warped = cv2.warpPerspective(img, H, (w, h))

    # Apply the "Background Removal" Crop (x1, y1, x2, y2)
    x1, y1, x2, y2 = crop_coords
    return warped[y1:y2, x1:x2]

def get_connector_crop(aligned_img, roi_config, margin=8):
    """(FOR TOP 1) Cuts out specific connector based on JSON percentages (x_min_rel)"""
    h, w, _ = aligned_img.shape
    x1 = int(roi_config['x_min_rel'] * w) - margin
    y1 = int(roi_config['y_min_rel'] * h) - margin
    x2 = int(roi_config['x_max_rel'] * w) + margin
    y2 = int(roi_config['y_max_rel'] * h) + margin

    # Safety check boundaries
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)

    return aligned_img[y1:y2, x1:x2]

def get_connector_crop_from_bbox(aligned_img, roi_config, margin=8):
    """(FOR TOP 2) Cuts out specific connector based on JSON 'bbox' (absolute pixels)"""
    h, w, _ = aligned_img.shape
    
    # Bbox is [x1, y1, x2, y2]
    x1, y1, x2, y2 = roi_config['bbox']
    
    # Apply margin
    x1 = x1 - margin
    y1 = y1 - margin
    x2 = x2 + margin
    y2 = y2 + margin
    
    # Safety check boundaries
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)
    
    return aligned_img[y1:y2, x1:x2]

In [10]:
# 1. Load Reference Image
# Try to find the reference image in the *original* 'TOP 1' data folder
ref_path = PROJECT_ROOT / "Data/TOP 1" / REFERENCE_IMG_NAME
if not ref_path.exists():
    print(f"WARNING: Reference image not found at {ref_path}, trying TOP 1 dir...")
    ref_path = RAW_DATA_DIR_1 / REFERENCE_IMG_NAME # Fallback
    if not ref_path.exists():
        print(f"FATAL ERROR: Reference image not found. Please check REFERENCE_IMG_NAME.")
        # This cell will fail, but that's intended if the ref is missing.

print(f"Using reference image: {ref_path}")
ref_img = load_image(ref_path)
ref_gray = cv2.cvtColor(ref_img, cv2.COLOR_BGR2GRAY)

# 2. Define the Background Crop (The fix we discussed)
# Coordinates: x1, y1, x2, y2
BG_CROP_BOX = (302, 288, 1883, 942)

# --- FIX: Process TOP 1 and TOP 2 in two separate loops ---

# --- LOOP 1: Process TOP 1 data with TOP 1_config --- 
print(f"\n--- Processing TOP 1 Data ---")
try:
    with open(ROI_CONFIG_PATH_1, 'r') as f:
        roi_configs_1 = json.load(f)
    print(f"Loaded {ROI_CONFIG_PATH_1}")
    
    all_images_1 = list(RAW_DATA_DIR_1.glob("*.png"))
    print(f"Processing {len(all_images_1)} images from TOP 1...")

    for img_path in tqdm(all_images_1, desc="Cropping TOP 1"):
        try:
            # Step A: Align & Background Crop
            img = load_image(img_path)
            aligned = align_image(img, ref_img, ref_gray, BG_CROP_BOX)
            if aligned is None:
                print(f" Alignment failed: {img_path.name}")
                continue

            # Save aligned image (to combined ALIGNED_DIR)
            cv2.imwrite(str(ALIGNED_DIR / img_path.name), aligned)

            # Step B: Cut out Connectors using TOP 1 config
            for roi in roi_configs_1:
                crop = get_connector_crop(aligned, roi, margin=8) # <-- Uses OLD function
                save_folder = CROPS_DIR / roi['name']
                save_folder.mkdir(exist_ok=True)
                cv2.imwrite(str(save_folder / img_path.name), crop)
        except Exception as e:
            print(f"X Error in TOP 1 loop for {img_path.name}: {e}")
            
except Exception as e:
    print(f"X Error processing TOP 1: {e}")


# --- LOOP 2: Process TOP 2 data with TOP 2_config --- 
print(f"\n--- Processing TOP 2 Data ---")
try:
    with open(ROI_CONFIG_PATH_2, 'r') as f:
        roi_configs_2_list = json.load(f)
    print(f"Loaded {ROI_CONFIG_PATH_2}")

    # --- FIX: Create a lookup map from the JSON data ---
    # The new JSON is not a template, it's a map.
    # We group all ROIs by their source image filename.
    print("Building ROI lookup map for TOP 2...")
    roi_map = {}
    for roi in roi_configs_2_list:
        # Get filename from "Data\aligned_top\20251118105546_TOP.png"
        img_name = Path(roi['source_image']).name 
        if img_name not in roi_map:
            roi_map[img_name] = []
        # Add the roi info (name and bbox)
        roi_map[img_name].append({
            "name": roi["roi_name"],
            "bbox": roi["bbox"]
        })
    print(f"Map built for {len(roi_map)} unique images.")
    # --- END FIX ---
    
    all_images_2 = list(RAW_DATA_DIR_2.glob("*.png"))
    print(f"Processing {len(all_images_2)} images from TOP 2...")

    for img_path in tqdm(all_images_2, desc="Cropping TOP 2"):
        try:
            # Step A: Align & Background Crop
            img = load_image(img_path)
            aligned = align_image(img, ref_img, ref_gray, BG_CROP_BOX)
            if aligned is None:
                print(f" Alignment failed: {img_path.name}")
                continue

            # Save aligned image (to combined ALIGNED_DIR)
            cv2.imwrite(str(ALIGNED_DIR / img_path.name), aligned)

            # --- FIX: Use the lookup map to find ROIs for *this* image ---
            if img_path.name not in roi_map:
                print(f"Warning: No ROIs found in JSON for image {img_path.name}. Skipping.")
                continue

            rois_for_this_image = roi_map[img_path.name]
            
            # Step B: Cut out Connectors using the new function
            for roi in rois_for_this_image:
                # Use the new function that reads 'bbox'
                crop = get_connector_crop_from_bbox(aligned, roi, margin=8) # <-- Uses NEW function
                
                save_folder = CROPS_DIR / roi['name']
                save_folder.mkdir(exist_ok=True)
                cv2.imwrite(str(save_folder / img_path.name), crop)
            # --- END FIX ---
        except Exception as e:
            print(f"X Error in TOP 2 loop for {img_path.name}: {e}")

except Exception as e:
    print(f"X Error processing TOP 2: {e}")
    import traceback
    traceback.print_exc() # Add detailed traceback

print(f"\n--- Combined Preprocessing Done! All crops are in {CROPS_DIR} ---")

Using reference image: E:\Project-Work\Data\TOP 1\20251106131917_TOP.png

--- Processing TOP 1 Data ---
Loaded E:\Project-Work\Preprocessing\roi_config.json
Processing 177 images from TOP 1...


Cropping TOP 1:   0%|          | 0/177 [00:00<?, ?it/s]


--- Processing TOP 2 Data ---
Loaded E:\Project-Work\Preprocessing\connector_crops.json
Building ROI lookup map for TOP 2...
Map built for 152 unique images.
Processing 152 images from TOP 2...


Cropping TOP 2:   0%|          | 0/152 [00:00<?, ?it/s]


--- Combined Preprocessing Done! All crops are in E:\Project-Work\Data\connectors ---


In [11]:
# --- NEW CELL: ONE-TIME PREPROCESSING ---
# This cell reads the color crops from the combined 'Data/connectors'
# and saves new Grayscale+CLAHE versions to 'Data/connectors_preprocessed'.

print("--- Starting One-Time Preprocessing (Grayscale + CLAHE) for COMBINED Data ---")

# 1. Create CLAHE object
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

# 2. Find all images in the original (color) crops directory (CROPS_DIR is now .../connectors)
image_paths = list(CROPS_DIR.glob('*/*.png'))
print(f"Found {len(image_paths)} images to preprocess...")

if not image_paths:
    print(f"X ERROR: No images found in {CROPS_DIR}. Did you run the previous cell first?")
else:
    for img_path in tqdm(image_paths, desc="Preprocessing crops"):
        try:
            # 1. Load image in color
            img = cv2.imread(str(img_path))
            if img is None:
                print(f"Warning: Could not read {img_path}. Skipping.")
                continue

            # 2. Convert to Grayscale
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

            # 3. Apply CLAHE
            clahe_img = clahe.apply(gray)
            
            # 4. Convert back to 3-Channel BGR
            # This is CRITICAL. The ResNet models (like wide_resnet50_2)
            # are pre-trained on 3-channel (RGB) images and expect
            # a 3-channel input, even if it's just grayscale repeated.
            clahe_img_bgr = cv2.cvtColor(clahe_img, cv2.COLOR_GRAY2BGR)

            # 5. Save to the new location, preserving structure
            # e.g., .../connectors/conn1/img.png -> .../connectors_preprocessed/conn1/img.png
            relative_path = img_path.relative_to(CROPS_DIR)
            new_save_path = PREPROCESSED_CROPS_DIR / relative_path
            
            new_save_path.parent.mkdir(parents=True, exist_ok=True)
            cv2.imwrite(str(new_save_path), clahe_img_bgr)

        except Exception as e:
            print(f"X Error processing {img_path.name}: {e}")

    print(f"--- Preprocessing Done! --- ")
    print(f"New dataset is in: {PREPROCESSED_CROPS_DIR}")

--- Starting One-Time Preprocessing (Grayscale + CLAHE) for COMBINED Data ---
Found 2961 images to preprocess...


Preprocessing crops:   0%|          | 0/2961 [00:00<?, ?it/s]

--- Preprocessing Done! --- 
New dataset is in: E:\Project-Work\Data\connectors_preprocessed


In [12]:
# --- MODIFIED CELL: SORT PREPROCESSED DATA ---
# This cell now reads from 'connectors_preprocessed' 
# and copies them into 'Final_Dataset'.

import shutil

# Outputs (FINAL_DATASET_DIR is .../Final_Dataset)
INSPECTOR_DATA = FINAL_DATASET_DIR / "inspector_by_connector"

# The SOURCE directory is the preprocessed one for the combined data
SOURCE_CROPS_DIR = PREPROCESSED_CROPS_DIR 

# 1. Clean and Recreate Structure
if INSPECTOR_DATA.exists():
    shutil.rmtree(INSPECTOR_DATA)

# 2. Read Labels (Combining both CSVs)
df1 = pd.read_csv(LABELS_CSV_1)
df2 = pd.read_csv(LABELS_CSV_2)
df = pd.concat([df1, df2], ignore_index=True)
print(f" Sorting {len(df)} images ({len(df1)} from TOP 1, {len(df2)} from TOP 2) into Connector folders...")

# 3. Sort Files
for _, row in tqdm(df.iterrows(), total=len(df)):
    connector = row['connector_name']
    label = str(row['label']).upper().strip()
    filename = row['filename']

    # Point to the source image in the preprocessed directory
    src_path = SOURCE_CROPS_DIR / connector / filename

    if not src_path.exists(): continue

    # Destination Paths (INSPECTOR_DATA is .../Final_Dataset/inspector_by_connector)
    conn_base = INSPECTOR_DATA / connector
    good_dir = conn_base / "good"
    bad_dir = conn_base / "bad"

    # Ensure folders exist immediately
    good_dir.mkdir(parents=True, exist_ok=True)
    bad_dir.mkdir(parents=True, exist_ok=True)

    unique_name = f"{connector}_{filename}"

    if label == 'OK':
        shutil.copy(src_path, good_dir / unique_name)
    elif label == 'KO':
        shutil.copy(src_path, bad_dir / unique_name)

print(" Dataset repaired. All 'bad' folders exist now.")

 Sorting 2925 images (1557 from TOP 1, 1368 from TOP 2) into Connector folders...


  0%|          | 0/2925 [00:00<?, ?it/s]

 Dataset repaired. All 'bad' folders exist now.


In [ ]:
# --- NEW TRAINING CELL (FOR COMBINED DATA) ---
# This cell replaces the truncated one and uses all our latest fixes.
# It will save models to the combined 'results' and 'Models/inspectors' folders.
torch.set_float32_matmul_precision('medium')
import shutil
import random
import torch
import cv2
import numpy as np
from pathlib import Path
import albumentations as A
from albumentations.pytorch import ToTensorV2
from anomalib.data import Folder
from anomalib.models import Patchcore
from anomalib.engine import Engine

# Global seed for reproducibility
SEED = 412

# --- 1. DEFINE OUR *FASTER* PREPROCESSING STEPS ---
imagenet_mean = (0.485, 0.456, 0.406)
imagenet_std = (0.229, 0.224, 0.225)

# Train Transform (with "jitter" augmentation)
train_transform = A.Compose([
    A.Affine(scale=(0.95, 1.05), translate_percent=(-0.05, 0.05), rotate=(-5, 5), p=0.5),
    A.Resize(256, 256), # Standard size for Patchcore
    A.Normalize(mean=imagenet_mean, std=imagenet_std),
    ToTensorV2(),
])

# Evaluation Transform (no "jitter" augmentation)
eval_transform = A.Compose([
    A.Resize(256, 256), # Standard size
    A.Normalize(mean=imagenet_mean, std=imagenet_std),
    ToTensorV2(),
])
# --- END NEW TRANSFORM DEFINITIONS ---

def augment_insufficient_data(conn_folder, min_bad=6):
    """Inflates 'bad' data to ensure stable test/val splits."""
    bad_dir = conn_folder / "bad"
    if not bad_dir.exists():
        return 0
    
    bad_images = list(bad_dir.glob("*.png"))
    count = len(bad_images)
    
    if count == 0:
        return 0
        
    if count < min_bad:
        print(f"    Inflating 'bad' data for stability (Original: {count})")
        needed = min_bad - count
        for i in range(needed):
            src = random.choice(bad_images)
            dst = bad_dir / f"{src.stem}_copy_{i}{src.suffix}"
            shutil.copy(src, dst)
            
        return len(list(bad_dir.glob("*.png")))
        
    return count

def train_multi_inspector():
    # 1. Setup paths
    # This path is correct, pointing to the combined FINAL_DATASET_DIR
    inspector_root = FINAL_DATASET_DIR / "inspector_by_connector"
    connectors = [d for d in inspector_root.iterdir() if d.is_dir()]
    print(f"Found {len(connectors)} connector types.")
    
    performance_report = []

    for conn_folder in connectors:
        conn_name = conn_folder.name
        print(f"\n--- Processing: {conn_name} ---")

        # 2. Inflate Data
        num_bad_final = augment_insufficient_data(conn_folder, min_bad=6)
        good_files = list((conn_folder / "good").glob("*.png"))
        num_good = len(good_files)
        print(f"    Data: {num_good} good, {num_bad_final} bad images.")

        if num_good < 5:
            print(f"    X Skipping {conn_name}: Not enough GOOD images.")
            continue
            
        # 3. Dynamic Batch Size
        safe_batch_size = 4 if num_good < 50 else 8
        abnormal_arg = "bad" if num_bad_final > 0 else None

        # 4. Setup DataModule
        # --- THIS IS THE FIX for BOTH errors ---
        # 1. Create the datamodule WITHOUT transform kwargs to avoid TypeError
        datamodule = Folder(
            name=conn_name,
            root=str(conn_folder),
            normal_dir="good",
            abnormal_dir=abnormal_arg,
            num_workers=0,
            train_batch_size=safe_batch_size,
            eval_batch_size=safe_batch_size,
            val_split_ratio=0.2,
            seed=SEED
            # NO 'train_transform' or 'val_transform' here
        )

        # 2. Manually set the transform attributes BEFORE .setup() is called by the engine
        # This fixes the 'Validation: 0/?' bug and works for all versions.
        datamodule.train_transform = train_transform
        datamodule.val_transform = eval_transform
        datamodule.test_transform = eval_transform
        # --- END FIX ---

        # 5. Setup Model
        try:
            model = Patchcore(
                backbone="wide_resnet50_2",
                pre_trained=True,
                coreset_sampling_ratio=0.1,
                layers=["layer1", "layer2"]
            )
        except Exception:
            print("    WideResNet failed (likely OOM), falling back to ResNet18")
            model = Patchcore(
                backbone="resnet18",
                pre_trained=True,
                coreset_sampling_ratio=0.1,
                layers=["layer1", "layer2"]
            )

        # 6. Setup Engine
        engine = Engine(
            max_epochs=1,
            default_root_dir=f"results/{conn_name}", # Save to combined 'results'
            accelerator="gpu",
            devices=1,
            num_sanity_val_steps=0,
            log_every_n_steps=1,
        )

        # 7. Train & Test
        try:
            engine.fit(datamodule=datamodule, model=model)
            
            if num_bad_final > 0:
                print(f"    Testing {conn_name}...")
                test_results = engine.test(datamodule=datamodule, model=model)
                try:
                    f1 = test_results[0].get('image_F1Score', 0.0)
                    auroc = test_results[0].get('image_AUROC', 0.0)
                except:
                    f1, auroc = 0.0, 0.0
            else:
                f1, auroc = -1, -1 # Mark as N/A

            performance_report.append({'name': conn_name, 'F1': f1, 'AUROC': auroc})

            # 8. Export Model
            output_path = PROJECT_ROOT / f"Models/inspectors/{conn_name}_model_torch" # Save to combined 'Models/inspectors'
            output_path.parent.mkdir(parents=True, exist_ok=True)
            model.export(output_path)
            
            print(f"    Saved: {conn_name}")

        except Exception as e:
            print(f"    X Failed processing {conn_name}: {e}")

    # Final Report
    print("\n--- SUMMARY ---")
    print(f"{'Connector': <15} | {'F1': <10} | {'AUROC': <10}")
    for res in performance_report:
        f1_s = f"{res['F1']:.2f}" if res['F1'] != -1 else "N/A"
        auroc_s = f"{res['AUROC']:.2f}" if res['AUROC'] != -1 else "N/A"
        print(f"{res['name']: <15} | {f1_s:<10} | {auroc_s:<10}")

# Run it
train_multi_inspector()


Found 9 connector types.

--- Processing: conn1 ---
    Inflating 'bad' data for stability (Original: 5)
    Data: 319 good, 6 bad images.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
You are using a CUDA device ('NVIDIA GeForce RTX 3070 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
e:\miniconda3\envs\pw_clean\lib\site-packages\lightning\pytorch\core\optimizer.py:183: `LightningModule.configure_optimizers` returned `None`, this fit will run with no optimizer

  | Name           | Type           | Params | Mode 
----------------------------------------------------------
0 | pre_processor  | PreProcessor   | 0      | train
1 | post_processor | PostProcessor  | 0      | train
2 | evaluator      | Evaluator      | 0      | train
3 | model          | PatchcoreModel | 4.1 M  | train

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]






















































































































































Selecting Coreset Indices.:   0%|          | 148/104857 [01:14<14:40:44,  1.98it/s]

Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

e:\miniconda3\envs\pw_clean\lib\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# --- LOAD MODELS (FOR COMBINED DATA) ---

# --- FIX: Set TRUST_REMOTE_CODE=1 --- 
import os
os.environ['TRUST_REMOTE_CODE'] = '1'
# --- END FIX ---

# --- Load Inspectors (Dictionary of Models) ---
inspector_models = {}
# Point to the combined models directory
inspector_dir = PROJECT_ROOT / "Models/inspectors"

print("Loading Inspector Models...")
for model_folder in inspector_dir.iterdir():
    if model_folder.is_dir():
        conn_name = model_folder.name.replace("_model_torch", "")
        model_file = model_folder / "weights/torch/model.pt"
        
        if model_file.exists():
            print(f"  Loading {conn_name}...")
            try:
                inspector_models[conn_name] = TorchInferencer(
                    path=model_file,
                    device="cuda"
                )
            except Exception as e:
                print(f"    X Error loading {conn_name}: {e}")

print(f"\nLoaded {len(inspector_models)} Inspector models.")

In [ ]:
# --- NEW INFERENCE FUNCTION (FOR COMBINED DATA) ---

from matplotlib.colors import Normalize

# --- Create a CLAHE object once for this function ---
clahe_processor = cv2.createCLAHE(clip_limit=2.0, tile_grid_size=(8, 8))


def visualize_prediction(image_path, inspector, intensity_threshold=0.6, min_area_threshold=100):
    """
    Visualizes the prediction with custom thresholding based on heatmap
    intensity and area.
    """
    
    # 1. Load Image (Original Color)
    img_path = str(image_path)
    image_color_bgr = cv2.imread(img_path)
    if image_color_bgr is None:
        print(f"Could not read {img_path}")
        return
    
    # Convert original color to RGB for Matplotlib plotting
    image_color_rgb = cv2.cvtColor(image_color_bgr, cv2.COLOR_BGR2RGB)

    # --- 3. PREPROCESS IMAGE FOR INSPECTOR ---
    gray = cv2.cvtColor(image_color_bgr, cv2.COLOR_BGR2GRAY)
    clahe_img = clahe_processor.apply(gray)
    preprocessed_bgr_for_model = cv2.cvtColor(clahe_img, cv2.COLOR_GRAY2BGR)
    # --- END PREPROCESSING ---

    # 4. Run Inspector
    inspector_result = inspector.predict(preprocessed_bgr_for_model) 

    # Extract maps and scores
    anomaly_map = inspector_result.anomaly_map
    pred_score = inspector_result.pred_score

    # Robust TENSOR to NUMPY conversion
    if isinstance(anomaly_map, torch.Tensor):
        anomaly_map = anomaly_map.detach().cpu().numpy()
    if isinstance(pred_score, torch.Tensor):
        score_val = pred_score.detach().cpu().item()
    else:
        score_val = float(pred_score)

    # 5. Create Visuals
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # Plot A: Original Image (in COLOR, for human eyes)
    axes[0].imshow(image_color_rgb)
    axes[0].set_title(f"Original Image")
    axes[0].axis("off")

    if anomaly_map is None:
        axes[1].text(0.5, 0.5, "No Map", ha='center')
        axes[1].axis("off")
        axes[2].text(0.5, 0.5, "No Map", ha='center')
        axes[2].axis("off")
        plt.show()
        return

    # Squeeze dimensions: (1, 1, H, W) -> (H, W) or (H, W) -> (H, W)
    if anomaly_map.ndim == 4:
        anomaly_map = np.squeeze(anomaly_map)
    elif anomaly_map.ndim == 3:
         anomaly_map = np.squeeze(anomaly_map, axis=0)
    
    # --- 6. CUSTOM THRESHOLDING LOGIC (Same as before) ---
    h, w = image_color_rgb.shape[:2]
    heatmap_resized = cv2.resize(anomaly_map, (w, h))

    # Normalize heatmap to 0-1
    min_val = heatmap_resized.min()
    max_val = heatmap_resized.max()
    heatmap_norm = heatmap_resized
    if max_val - min_val > 1e-6:
        heatmap_norm = (heatmap_resized - min_val) / (max_val - min_val)
    
    intensity_mask = (heatmap_norm > intensity_threshold).astype(np.uint8)
    anomaly_pixel_count = int(np.sum(intensity_mask))
    verdict = "DISCONNECTED" if anomaly_pixel_count > min_area_threshold else "CONNECTED"
    color = "red" if verdict == "DISCONNECTED" else "green"

    # --- 7. UPDATE VISUALS (Same as before) ---
    axes[1].imshow(heatmap_norm, cmap="hot", vmin=0, vmax=1)
    axes[1].set_title(f"Normalized Map (Orig Score: {score_val:.4f})\nIntensity > {intensity_threshold}")
    axes[1].axis("off")

    # Overlay on the COLOR image
    overlay_img = np.zeros_like(image_color_rgb, dtype=np.uint8)
    overlay_img[intensity_mask == 1] = [255, 0, 0] # Red mask

    axes[2].imshow(image_color_rgb) # Show original color
    axes[2].imshow(overlay_img, alpha=0.6) 
    axes[2].set_title(f"Verdict: {verdict}\n(Area: {anomaly_pixel_count} px > {min_area_threshold} px?)", 
                        color=color, fontweight="bold")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# --- NEW TEST RUN CELL (FOR COMBINED DATA) ---

import random

# --- 1. DEFINE YOUR PER-CONNECTOR TEST PARAMETERS --- 

# Set default thresholds for any connector NOT listed below
DEFAULT_THRESHOLDS = {"intensity": 0.6, "area": 100}

# Define custom thresholds for specific connectors
# You will need to tune these values by looking at the output
# These are the same as before, but you may need to re-tune them
# now that the model is trained on more data.
CONNECTOR_THRESHOLDS = {
    "conn1": {"intensity": 0.65, "area": 150},
    "conn2": {"intensity": 0.7, "area": 100},
    "conn3": {"intensity": 0.8, "area": 450}, 
    "conn4": {"intensity": 0.7, "area": 80},
    "conn5": {"intensity": 0.6, "area": 200},
    "conn6": {"intensity": 0.7, "area": 120},
    "conn7": {"intensity": 0.7, "area": 120},
    "conn8": {"intensity": 0.8, "area": 450},
    "conn9": {"intensity": 0.6, "area": 150},
}

MAX_GOOD_IMAGES_TO_TEST = 10 # Number of good images to test per connector

print(f"--- Starting Full Test Run ---")
print(f"Using per-connector thresholds. Default is: {DEFAULT_THRESHOLDS}")
print(f"Testing up to {MAX_GOOD_IMAGES_TO_TEST} good images per connector.")

# --- 2. FIND ALL CONNECTORS TO TEST ---
try:
    # We find the connectors from the *combined* training data folders
    INSPECTOR_DATA = FINAL_DATASET_DIR / "inspector_by_connector"
    all_connector_folders = [d for d in INSPECTOR_DATA.iterdir() if d.is_dir()]

    if not all_connector_folders:
        print(f"\n[ERROR] No connector folders found in {INSPECTOR_DATA}")

    # --- 3. LOOP THROUGH EACH CONNECTOR ---
    for i, test_conn_folder in enumerate(all_connector_folders):
        connector_name = test_conn_folder.name
        
        print(f"\n" + "="*50)
        print(f"TESTING CONNECTOR {i+1}/{len(all_connector_folders)}: {connector_name}")
        print("="*50)

        # Get the specific inspector model for this connector
        if connector_name not in inspector_models:
            print(f"  [Error] No inspector model found for '{connector_name}'. Skipping.")
            continue
        
        inspector_model = inspector_models[connector_name]
        
        # --- Get the correct thresholds for this connector ---
        thresholds = CONNECTOR_THRESHOLDS.get(connector_name, DEFAULT_THRESHOLDS)
        current_intensity = thresholds["intensity"]
        current_area = thresholds["area"]
        print(f"  Using Thresholds: Intensity > {current_intensity}, Area > {current_area}")

        # Get image paths from the sorted training data
        bad_images_paths = list((test_conn_folder / "bad").glob("*.png"))
        good_images_paths = list((test_conn_folder / "good").glob("*.png"))

        # --- 4. RUN VISUALIZATION FOR 'BAD' IMAGES ---
        if not bad_images_paths:
            print("\n  No 'bad' images found to test for this connector.")
        else:
            print(f"\n  --- Testing {len(bad_images_paths)} 'BAD' image(s) ---")
            for j, test_img_path in enumerate(bad_images_paths):
                print(f"  Testing bad image {j+1}/{len(bad_images_paths)}: {test_img_path.name}")
                
                # --- Find the ORIGINAL COLOR image path ---
                stem = test_img_path.stem
                suffix = test_img_path.suffix
                name_without_prefix = stem.replace(f"{connector_name}_", "", 1)
                base_name = name_without_prefix.split("_copy_")[0]
                original_filename = base_name + suffix
                
                # 4. Point to the *original color crop* directory
                original_color_crop_path = CROPS_DIR / connector_name / original_filename
                
                if original_color_crop_path.exists():
                     visualize_prediction(original_color_crop_path, 
                                         inspector_model,
                                         intensity_threshold=current_intensity,
                                         min_area_threshold=current_area)
                else:
                    print(f"    [Error] Could not find original COLOR crop: {original_color_crop_path}")

        # --- 5. RUN VISUALIZATION FOR 'GOOD' IMAGES ---
        if not good_images_paths:
            print("\n  No 'good' images found to test for this connector.")
        else:
            random.shuffle(good_images_paths)
            good_images_to_test = good_images_paths[:MAX_GOOD_IMAGES_TO_TEST]
            
            print(f"\n  --- Testing {len(good_images_to_test)} (of {len(good_images_paths)}) random 'GOOD' image(s) ---")
            for j, test_img_path in enumerate(good_images_to_test):
                print(f"  Testing good image {j+1}/{len(good_images_to_test)}: {test_img_path.name}")
                
                # --- Find the ORIGINAL COLOR image path ---
                stem = test_img_path.stem
                suffix = test_img_path.suffix
                name_without_prefix = stem.replace(f"{connector_name}_", "", 1)
                base_name = name_without_prefix.split("_copy_")[0]
                original_filename = base_name + suffix
                original_color_crop_path = CROPS_DIR / connector_name / original_filename

                if original_color_crop_path.exists():
                    visualize_prediction(original_color_crop_path, 
                                         inspector_model,
                                         intensity_threshold=current_intensity,
                                         min_area_threshold=current_area)
                else:
                    print(f"    [Error] Could not find original COLOR crop: {original_color_crop_path}")

    print("\n" + "="*50)
    print("--- Full Test Run Complete ---")
    print("="*50)


except Exception as e:
    print(f"\n[CRITICAL ERROR] Test run failed.")
    print(f"Error during setup. Did you run the previous cells (PROJECT_ROOT, inspector_models)?")
    print(f"Details: {e}")
    import traceback
    traceback.print_exc()
